# causality 

In [3]:

import numpy as np
import tigramite.data_processing as pp
import pandas as pd
import matplotlib.pyplot as plt
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.independence_tests.robust_parcorr import RobustParCorr
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.stattools import pacf
from statsmodels.tsa.stattools import ccf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL
import statsmodels.api as sm
from multiprocessing import Array, Pool
from functools import partial
import os
from tqdm import tqdm
from itertools import product


In [2]:
def get_tau_ccf(df, var1, var2, max_lags=5):
    """
    Calculate the cross-correlation function (CCF) between two variables and determine tau_max.

    Parameters:
    - df: pandas DataFrame containing the time series data to be analyzed
    - var1: str, column name of the first variable
    - var2: str, column name of the second variable
    - max_lags: int, maximum number of lags to consider
    - alpha: float, significance level (default 0.05)

    Returns:
    - tau_max: int, the significant maximum lag value
    """
    ccf_values = ccf(df[var1], df[var2])
    ccf_values = ccf_values[:max_lags]
    n = len(df[var1])
    significance_threshold = 1.96 / np.sqrt(n)  
    # Find the significant maximum lag
    significant_lags = np.where(np.abs(ccf_values) > significance_threshold)[0]
    if significant_lags is not None:
        tau_max = significant_lags[-1] if significant_lags.size > 0 else None
    
    if tau_max is not None:
        if tau_max >= 12:
            tau_max = 5
        else:
            tau_max = tau_max
    elif tau_max is None:
        tau_max = 5
    return tau_max


def process_var_pair(pair, graph, val_matrix, q_matrix, variables, tau_max):
    m, n = pair
    pair_links = []
    
    for tau in range(0, tau_max + 1):
        if graph[m, n, tau] == '-->':
            link_info = {
                'cause': variables[m],
                'effect': variables[n],
                'lag' : tau,
                'symbol':'-->',
                'mci':val_matrix[m, n, tau],
                'p':q_matrix[m, n, tau]
            }
            pair_links.append(link_info)
        
    if m <= n:
        if graph[m, n, 0] == 'o-o':
            link_info = {
                'cause':variables[m],
                'effect':variables[n],
                'lag':0,
                'symbol':'o-o',
                'mci':val_matrix[m, n, 0],
                'p':q_matrix[m, n, 0]
            }
            pair_links.append(link_info)
        
        elif graph[m, n, 0] == 'x-x':
            link_info = {
                'cause':variables[m],
                'effect':variables[n],
                'lag':0,
                'symbol':'x-x',
                'mci':val_matrix[m, n, 0],
                'p':q_matrix[m, n, 0]
            }
            pair_links.append(link_info)

    return pair_links

## ALAN_DFS

In [ ]:
base_path = r"H:\PCMCI+\ALAN_DFS\data"
out_path = r"H:\PCMCI+\ALAN_DFS\result"   
alan_file = os.path.join(base_path, "ALAN.xlsx")
DFS_file = os.path.join(base_path, "DFS.xlsx")
prec_file = os.path.join(base_path, "prec.xlsx")
temp_file = os.path.join(base_path, "temp.xlsx")
solar_file = os.path.join(base_path, "solar.xlsx")

alan_df = pd.read_excel(alan_file, index_col=0)
DFS_df = pd.read_excel(DFS_file, index_col=0)
prec_df = pd.read_excel(prec_file, index_col=0)
temp_df = pd.read_excel(temp_file, index_col=0)
solar_df = pd.read_excel(solar_file, index_col=0)

city_ids = alan_df.index

years = [str(year) for year in range(2001, 2023)]
alan_df.columns = alan_df.columns.astype(str)
DFS_df.columns = DFS_df.columns.astype(str)
prec_df.columns = prec_df.columns.astype(str)
temp_df.columns = temp_df.columns.astype(str)
solar_df.columns = solar_df.columns.astype(str)

all_results = []

for city_id in tqdm(city_ids, desc="Processing Cities"):
    try:
        # Prepare city data
        city_data = pd.DataFrame({
            'ALAN': alan_df.loc[city_id, years].values.astype(float),
            'DFS': DFS_df.loc[city_id, years].values.astype(float),
            'TEMP': temp_df.loc[city_id, years].values.astype(float),
            'PREC': prec_df.loc[city_id, years].values.astype(float),
            'SOLAR': solar_df.loc[city_id, years].values.astype(float),
        })
       
        # Data validation
        checked = city_data.values
        if checked is None:
            continue

        city_data = pd.DataFrame(checked, columns=['ALAN', 'DFS', 'TEMP', 'PREC', 'SOLAR'])

        # Automatically determine tau_max
        tau_max = get_tau_ccf(city_data, 'DFS', 'ALAN')

        tigramite_data = pp.DataFrame(city_data.values, datatime={0: np.arange(len(city_data))}, var_names=city_data.columns.tolist())

        # Run PCMCI+
        pcmci = PCMCI(dataframe=tigramite_data, cond_ind_test=RobustParCorr())
        results = pcmci.run_pcmciplus(tau_min=0, tau_max=tau_max, pc_alpha=0.1, contemp_collider_rule='majority',
                                     conflict_resolution=True, reset_lagged_links=True)

        graph = results['graph']
        val_matrix = results['val_matrix']
        q_matrix = pcmci.get_corrected_pvalues(results['p_matrix'])

        # Extract causal relationships
        var_pairs = list(product(range(len(city_data.columns)), repeat=2))
        process_func = partial(
            process_var_pair,
            graph=graph,
            val_matrix=val_matrix,
            q_matrix=q_matrix,
            variables=city_data.columns.tolist(),
            tau_max=tau_max
        )

        causal_links = []
        for pair in var_pairs:
            causal_links.extend(process_func(pair))

        # Filter for ALAN --> DFS relationships
        alan_to_DFS_links = [d for d in causal_links if d['cause'] == 'ALAN' and d['effect'] == 'DFS']
        for d in alan_to_DFS_links:
            d['city_id'] = city_id

        all_results.extend(alan_to_DFS_links)

    except Exception as e:
        print(f"Error processing city {city_id}: {e}")
        continue

# Save all results to Excel
result_df = pd.DataFrame(all_results)
save_path = os.path.join(out_path, "ALAN_DFS.xlsx")
result_df.to_excel(save_path, index=False)

print(f"Analyzed {len(city_ids)} cities. Results saved to: {save_path}")